# B2-021 — Session 3: U-Net Segmentation and Skip Connections

*90 minutes.*

**Program layer:** Round 2 extension  
**Compute:** `compute.policy: cpu` · seed `20260901`  
**Qualified prerequisites:** `book1:F1-scientific-python`, `book1:F3-matrices`, `book1:C6-pytorch`, `book1:C7-cnn-transfer`, `book1:C11-neural-training`, `B2-019-attention-transformers`, `B2-020-language-transformers`  
**Remediation links actually used:** [book1:F1-scientific-python](../../../../book1/units/F1-scientific-python/lesson.ipynb), [book1:F3-matrices](../../../../book1/units/F3-matrices/lesson.ipynb), [book1:C6-pytorch](../../../../book1/units/C6-pytorch/lesson.ipynb), [book1:C7-cnn-transfer](../../../../book1/units/C7-cnn-transfer/lesson.ipynb), [book1:C11-neural-training](../../../../book1/units/C11-neural-training/lesson.ipynb), [B2-019-attention-transformers](../../B2-019-attention-transformers/lesson.ipynb), [B2-020-language-transformers](../../B2-020-language-transformers/lesson.ipynb).

**Data boundary:** CPU only; use literal unit data and standard library/NumPy/Torch only. Do not use the web, downloads, external datasets, model hubs, pretrained models, checkpoints, or student data.


## 1. Dense prediction and mask contract

Segmentation maps every input pixel to a class. Images are CPU float32 `(B,1,8,8)` and masks are int64 `(B,8,8)` with class IDs `0` or `1`. Logits are float32 `(B,2,8,8)` and remain spatially aligned with the input. Do not one-hot the target for Torch pixel cross-entropy.

**Checkpoint 1A.** Which tensor omits the channel axis?

**Checkpoint 1B.** What does logit channel 1 mean?

**Collected answers.** The int64 target mask; it is the foreground-class logit at each pixel.


## 2. Encoder resolution ledger

A one-down encoder uses a padded 3x3 convolution from `(B,1,8,8)` to skip feature `s1=(B,4,8,8)`, then 2x2 max pooling to `(B,4,4,4)`, then a padded 3x3 bottleneck to `(B,8,4,4)`. Padding 1 preserves resolution across each 3x3 convolution; pooling halves it.

**Checkpoint 2A.** What is stored for the skip?

**Checkpoint 2B.** What are bottleneck spatial dimensions?

**Collected answers.** Store `(B,4,8,8)` before pooling; bottleneck is 4x4.


## 3. Decoder, skip concatenation, and output alignment

A 2x2 stride-2 `ConvTranspose2d(8,4,2,2)` restores `(B,4,8,8)`. Concatenate with `s1` on channel axis 1, yielding `(B,8,8,8)`, then fuse and project with a 1x1 head to `(B,2,8,8)`. Concatenation requires equal batch, height, and width; never crop silently under this exact even-sized ledger.

**Checkpoint 3A.** Which axis doubles at the skip?

**Checkpoint 3B.** Why must the output be 8x8?

**Collected answers.** Channels double from 4+4; pixel logits must align one-for-one with the 8x8 mask.


## 4. Worked skip-concat and Dice calculation

Worked shapes: decoder `(1,4,8,8)` plus skip `(1,4,8,8)` gives `(1,8,8,8)`. For predicted foreground pixels `{0,1,2}` and target `{1,2,3}`, intersection is 2 and sizes are 3 and 3. With smoothing `eps=1e-7`, `Dice=(2*2+eps)/(3+3+eps)≈2/3`; `IoU=(2+eps)/(3+3-2+eps)≈1/2`. State whether metrics use hard argmax masks or soft probabilities; this unit's reported metric uses hard masks.

**Checkpoint 4A.** What differs in the two denominators?

**Checkpoint 4B.** Why pin smoothing?

**Collected answers.** Dice sums the two set sizes; IoU subtracts intersection. Pinned smoothing makes empty-set behavior and tolerance reproducible.


## 5. Masked pixel cross-entropy

For an optional boolean valid mask `(B,8,8)`, compute unreduced `F.cross_entropy(logits,target,reduction='none')`, select valid pixels, and take their mean. Reject an all-false mask rather than returning NaN. Metrics use the same valid mask. This teaches `reduction='none'`, boolean indexing, and explicit denominators before p11.

**Checkpoint 5A.** Why not multiply then call `.mean()` over all pixels?

**Checkpoint 5B.** What happens for zero valid pixels?

**Collected answers.** That would dilute the loss with invalid zeros; raise `ValueError`.


## 6. Fixed-seed tiny segmentation train/eval trace

This actual lesson-only mini-trace trains a 1x1 two-class pixel head on two literal 4x4 masks and evaluates one distinct held-out 4x4 mask. It uses seed `20260901`, 30 AdamW updates at `lr=.1`, and output shape `(1,2,4,4)`; it does not import `vision_fixture.py` and is deliberately not the p19 U-Net.

| update | train pixel CE | held-out pixel CE | held-out foreground Dice |
|---:|---:|---:|---:|
| 0 | 0.741990 | 0.741990 | 0.00 |
| 1 | 0.687156 | 0.687156 | 0.00 |
| 15 | 0.257748 | 0.257748 | 1.00 |
| 30 | 0.094345 | 0.094345 | 1.00 |

The update-1 row shows why lower CE need not immediately change a hard argmax Dice score. The held-out mask is evaluated under `no_grad` and never contributes to the gradient or threshold choice. p19 separately tests skip concatenation, 8x8 alignment, and a different 16-update model; no p19 final answer value is disclosed here.

**Checkpoint 6A.** How do the lesson demonstration and p19 update-count/lr protocols differ?

**Checkpoint 6B.** Which p19 intermediate proves the skip is used?

**Collected answers.** The lesson-only demonstration uses exactly 30 updates at `lr=.1`; p19 later requires exactly 16 updates at `lr=.03`. In p19, the concatenated `(B,8,8,8)` tensor proves aligned encoder information enters the decoder.


## 7. Worked laboratory: prove every activation and alignment

Treat the U-Net as an executable ledger rather than a picture. The encoder convolution maps `(B,1,8,8)` to `(B,4,8,8)` and is followed by ReLU; this activated tensor is the skip. Max pooling maps it to `(B,4,4,4)`. The bottleneck convolution maps four channels to eight at 4x4 and is followed by ReLU. The transposed convolution maps `(B,8,4,4)` to `(B,4,8,8)` and is also followed by ReLU. These three activation sites are required, not stylistic suggestions.

Concatenate decoder first and skip second on channel axis to obtain `(B,8,8,8)`. The fuse convolution maps eight channels to four but has no following ReLU. The head immediately maps those signed fused features to two raw logit channels, also without ReLU. This final distinction matters: a fuse ReLU can destroy negative evidence needed to separate foreground and background. On the literal fixture and fixed sixteen-update protocol, that mutant can lower CE while producing essentially zero foreground Dice, which is why both loss and hard overlap are checked.

The first code cell executes only the shape path with seeded modules. It asserts equality before concatenation rather than cropping or interpolating. If the decoder were 7x8, the first violation is spatial mismatch at concatenation. Repair the architecture or input padding so both paths naturally reach 8x8; never resize the int64 target with bilinear interpolation. That would invent fractional class boundaries and conceal the actual network defect.

For pixel cross-entropy, logits remain `(B,2,H,W)` and targets remain int64 `(B,H,W)`. With a validity mask, compute unreduced CE, select valid pixels, and divide by the number selected. Multiplying invalid pixels by zero and averaging over every pixel changes the denominator. Reject an empty mask explicitly. For hard foreground Dice, take `argmax` first, intersect boolean foreground sets, and use the pinned epsilon in numerator and denominator.

The second code cell compares Dice and IoU on a literal three-pixel prediction and target. Both share the same intersection, but Dice denominator adds set sizes while IoU uses union. In highly imbalanced masks, background accuracy can exceed ninety percent even when foreground Dice is zero. Therefore p19 requires both a held-out CE improvement and a foreground Dice threshold.

Audit the training trace in order. Baseline evaluation precedes any optimizer step. Each of sixteen updates uses only all four training images in stored order. Held-out images are evaluated with `no_grad`; they never tune thresholds or augmentations. Repeated `compute_loss` calls after a step must remain valid because fingerprints bind the unchanged feature and target arrays, not mutable model parameters. If validation depends on logits or weights, it has crossed the wrong seam.

Before leaving the session, explain the skip’s purpose without saying merely “more detail.” Pooling compresses spatial location; the high-resolution encoder tensor preserves boundary-aligned evidence. Concatenation lets the fuse convolution choose jointly from decoded semantic features and preserved local features. The returned concat probe proves that the skip entered at the promised resolution and channel order.

**Checkpoint 7A.** List every required ReLU and the two locations where ReLU is prohibited.

**Checkpoint 7B.** What does returning the `(B,8,8,8)` concatenation certify?

**Collected answers.** Apply ReLU after encoder, bottleneck, and decoder; apply none after fuse or head. The returned concatenation certifies exact spatial alignment and that decoder plus high-resolution skip entered the fuse stage in the pinned order.


In [ ]:
import torch
from torch import nn
from torch.nn import functional as F

torch.manual_seed(20260901)
x = torch.randn(2, 1, 8, 8)
encoder, bottleneck = nn.Conv2d(1,4,3,padding=1), nn.Conv2d(4,8,3,padding=1)
decoder = nn.ConvTranspose2d(8,4,2,stride=2)
skip = F.relu(encoder(x))
deep = F.relu(bottleneck(F.max_pool2d(skip, 2)))
up = F.relu(decoder(deep))
assert up.shape == skip.shape == (2,4,8,8)
concat = torch.cat((up, skip), dim=1)
print(tuple(skip.shape), tuple(deep.shape), tuple(concat.shape))


In [ ]:
pred = {0, 1, 2}
truth = {1, 2, 3}
eps = 1e-7
intersection = len(pred & truth)
dice = (2*intersection + eps) / (len(pred) + len(truth) + eps)
iou = (intersection + eps) / (len(pred | truth) + eps)
assert abs(dice - 2/3) < 1e-6 and abs(iou - 1/2) < 1e-6
print({'intersection': intersection, 'dice': dice, 'iou': iou})


## 8. Common pitfalls and Exam connections

Common pitfalls are concatenating across width, using float class-index masks, comparing different resolutions, resizing targets with a smoothing interpolation, including held-out masks in augmentation or threshold choice, and reporting accuracy on mostly background pixels. Exam connections ask for the earliest mismatch in a resolution trace or for Dice/IoU from literal sets.

**Checkpoint 8A.** Which practice repairs a spatial mismatch?

**Checkpoint 8B.** Which metric resists background dominance?

**Collected answers.** p23; foreground Dice (and IoU) reveals overlap better than raw pixel accuracy.


## 9. Forward-only deeper topic: multiscale segmentation

Forward-only deeper topic: full U-Nets repeat down/up stages, may use residual blocks, deep supervision, or learned alignment for irregular sizes. These extensions are not assessed and cannot justify silent interpolation in this unit. The executable contract stays one downsample, one upsample, exact 8x8 alignment, and one skip.

This final section is forward-only and not assessed.

**Checkpoint 9A.** May a solver add a second downsample?

**Checkpoint 9B.** What alignment rule remains required?

**Collected answers.** No; decoder and skip must already share exact 8x8 spatial dimensions.
